# 04 · Convolutional networks on the trace curves

*Level: intermediate. Guide Section 8. Code: `models.CurveCNN`, `data.curve_tensor`, `data.log_frequency`.*

**Idea.** The hand-made features are numbers *we* chose. A convolutional network gets the **curves** themselves (Allan deviation,
noise spectrum, pulse shapes) and learns which shapes matter, so it can find effects missing from our list.

In [1]:
from tutorial_utils import *

import torch
from ml.benchmark import ALL_CURVES, inputs_for

## 1. The curves

Each example comes with 7 curves of the traces at θ and 7 of the reference traces.

In [2]:
df, curves, Y, groups = toy_scan(60, seed=1)
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for r in range(4):
    ax[0].loglog(synthetic.ALLAN_N, curves["noise: Allan deviation"][r]); ax[1].semilogy(curves["noise: CDS spectrum"][r]); ax[2].plot(curves["skip: mean shape"][r])
ax[0].set_title("Allan deviation (one line per amplifier)"); ax[1].set_title("CDS noise spectrum (linear frequency)"); ax[2].set_title("mean skip waveform")
plt.tight_layout(); plt.show()

## 2. Convolution by hand

A filter $w[k]$ slides along a curve $x[t]$:

$$(w \star x)[t] = \sum_k w[k]\, x[t+k] + b .$$

The output is large where the curve locally looks like the filter. In a CNN, the filter weights are learned. The same filter is used at every
position, so a feature is found wherever it appears.

In [3]:
xcurve = np.zeros(60); xcurve[20:24] = [0.5, 1, 1, 0.5]; xcurve[40:44] = [0.5, 1, 1, 0.5]
w = np.array([0.25, 0.5, 0.5, 0.25])
by_hand = np.array([np.sum(w * xcurve[t:t + 4]) for t in range(len(xcurve) - 3)])
by_torch = torch.nn.functional.conv1d(torch.tensor(xcurve)[None, None], torch.tensor(w)[None, None])[0, 0].numpy()
print("hand-made and PyTorch results equal:", np.allclose(by_hand, by_torch))
plt.figure(figsize=(7, 2.5)); plt.plot(xcurve, label="curve"); plt.plot(np.arange(len(by_hand)) + 1.5, by_hand, label="filter output"); plt.legend(); plt.show()
print("max pooling (pairs):", np.max(by_hand[:20].reshape(-1, 2), axis=1).round(2))

hand-made and PyTorch results equal: True
max pooling (pairs): [0.   0.   0.   0.   0.   0.   0.   0.   0.12 1.  ]


## 3. The representation: a logarithmic frequency axis

On a linear frequency axis, the lowest frequencies (where slow noise that does not average down lives) are a handful of points out of
hundreds. `data.log_frequency` resamples the spectrum so that **every decade gets the same number of points**.

In [4]:
spec = curves["noise: CDS spectrum"][0]
logf = data.log_frequency(spec)[0]
f = np.linspace(0, 1, len(spec))
print(f"points below 1 % of the top frequency: linear axis {np.sum((f > 0) & (f < 0.01))}, log axis {np.sum(np.r_[0, np.geomspace(f[1], 1, len(spec))[:-1]] < 0.01) - 1}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].semilogy(spec); ax[0].set_title("as computed (linear frequency)")
ax[1].semilogy(logf); ax[1].set_title("resampled on a log-frequency axis")
plt.tight_layout(); plt.show()

points below 1 % of the top frequency: linear axis 2, log axis 40


## 4. The CNN on the toy CCD: linear against logarithmic frequency

Train on one scan (120 packets) and test on a new scan, once with each representation. Our `CurveCNN` has:
* 3 convolution layers (7, 7 and 5 taps; 16, 32 and 32 filters), with pooling;
* the settings appended after the convolutions;
* a dense layer that outputs a mean and a variance;
* an ensemble (2 members here, to keep it quick).

In [5]:
dfa, ca, Ya, ga = toy_scan(120, seed=1); dfb, cb, Yb, gb = toy_scan(60, seed=2)
for logf_ in (False, True):
    Xa, _ = inputs_for(dfa, ca, ("theta",), True, ALL_CURVES, log_frequency_spectra=logf_)
    Xb, _ = inputs_for(dfb, cb, ("theta",), True, ALL_CURVES, log_frequency_spectra=logf_)
    cnn = models.CurveCNN(n_models=2).fit(Xa, Ya)
    m, s = cnn.predict(Xb)
    _ = report(f"CNN, {'log' if logf_ else 'linear'}-frequency spectra", Yb, m, s)

CNN, linear-frequency spectra            sigma_e:  10.1 % (cover 0.73)   lambda:   6.0 % (cover 0.65)   gain:   1.2 % (cover 0.78)


CNN, log-frequency spectra               sigma_e:   4.8 % (cover 0.79)   lambda:   8.1 % (cover 0.57)   gain:   1.4 % (cover 0.81)


In the project's full tests (240 packets) the logarithmic axis cut the CNN's σe error from 10.7 % to 1.8 %: the largest single gain we found.
**How the data are presented often matters more than the model.**

## When to use it

* **Good:** curves are available and there are enough labelled packets (a few hundred); to find effects the features miss.
* **Bad:** few labels. Then pre-train (12) or put the CNN inside a GP (13).

Next: **05 · Gaussian processes**, the workhorse of the project.